# Rendimiento y riesgo de un portafolio

Librerías

In [1]:
import pandas as pd
import numpy as np
import yfinance as yf

Definimos tickers de las acciones a trabajar

In [2]:
tickers = ['NVDA', 'WMT', 'AAPL', 'AMZN', 'MSFT']

Descargamos los precios de cierre

In [3]:
prices = yf.download(tickers, start='2024-01-01', end='2026-08-26')['Close']
prices

[*********************100%***********************]  5 of 5 completed


Ticker,AAPL,AMZN,MSFT,NVDA,WMT
Date,,,,,
2024-01-02,183.404037,149.929993,363.117950,48.082535,51.641460
2024-01-03,182.030762,148.470001,362.853546,47.484592,51.644703
2024-01-04,179.718933,144.570007,360.249146,47.912842,51.145439
2024-01-05,178.997757,145.240005,360.063202,49.009888,50.805031
2024-01-08,183.324966,149.100006,366.858063,52.160286,51.304283
...,...,...,...,...,...
2026-08-19,316.829987,265.839996,483.399994,217.559998,114.027023
2026-08-20,311.299988,260.109985,481.149994,216.850006,103.591995
2026-08-21,309.350006,258.630005,483.239990,214.720001,103.699997


## Rendimeinto del portafolio

$$
E(R_p)=\sum_{i=1}^{n} w_i E(R_i)
$$

Calcular rendimientos diarios individuales

In [4]:
rets = (prices / prices.shift() - 1).dropna()
rets

Ticker,AAPL,AMZN,MSFT,NVDA,WMT
Date,,,,,
2024-01-03,-0.007488,-0.009738,-0.000728,-0.012436,0.000063
2024-01-04,-0.012700,-0.026268,-0.007178,0.009019,-0.009667
2024-01-05,-0.004013,0.004634,-0.000516,0.022897,-0.006656
2024-01-08,0.024175,0.026577,0.018871,0.064281,0.009827
2024-01-09,-0.002263,0.015225,0.002936,0.016975,0.006698
...,...,...,...,...,...
2026-08-19,0.021933,0.024629,0.005564,-0.009921,-0.007812
2026-08-20,-0.017454,-0.021554,-0.004655,-0.003263,-0.091514
2026-08-21,-0.006264,-0.005690,0.004344,-0.009822,0.001043


Rendimientos promedio / rendimiento esperado (anual)

In [5]:
mu = rets.mean() * 252
mu * 100

Ticker
AAPL    23.763398
AMZN    26.355383
MSFT    15.092610
NVDA    68.200414
WMT     29.867533
dtype: float64

Rendimiento del portafolio

In [6]:
w_port = np.array([0.2, 0.2, 0.2, 0.2, 0.2]) # Pesos de los activos
w_port

array([0.2, 0.2, 0.2, 0.2, 0.2])

In [7]:
r_port = sum(w_port * mu) # Matemáticamente se realiza un producto punto 
r_port

0.32655867479240813

Si fuera con 2 activos es fácil, pero con más, se puede volver complicado, así que podemos usar lo siguiente:

$$
E(R_p) = w^T R
$$

## Riesgo del portafolio
$$
\sigma_p^2
=
\sum_{i=1}^{n}
\sum_{j=1}^{n}
X_i X_j \sigma_{i,j}
$$

Aquí no podemos considerar el riesgo de un activo por sí solo, debemos calcular también si hay relación entre los activos. Es decir, hay que considerar el efecto de interacción entre los activos.

In [8]:
rets.var()

Ticker
AAPL    0.000305
AMZN    0.000423
MSFT    0.000287
NVDA    0.000922
WMT     0.000218
dtype: float64

In [9]:
annual_var = rets.var() * 252
annual_var

Ticker
AAPL    0.076948
AMZN    0.106547
MSFT    0.072232
NVDA    0.232222
WMT     0.054965
dtype: float64

In [10]:
# Covarianzas
cov = rets.cov()
cov

Ticker,AAPL,AMZN,MSFT,NVDA,WMT
Ticker,,,,,
AAPL,0.000305,0.000130,0.000099,0.000159,0.000062
AMZN,0.000130,0.000423,0.000185,0.000276,0.000046
MSFT,0.000099,0.000185,0.000287,0.000219,0.000019
NVDA,0.000159,0.000276,0.000219,0.000922,0.000014
WMT,0.000062,0.000046,0.000019,0.000014,0.000218


De igual modo, esto es fácil si hay 2 activos, pero con más activos, se vuelve complicado, por lo que se resume a esta formula:

$$
\sigma_p^2 = w^T \Sigma w
$$

In [11]:
cov

Ticker,AAPL,AMZN,MSFT,NVDA,WMT
Ticker,,,,,
AAPL,0.000305,0.000130,0.000099,0.000159,0.000062
AMZN,0.000130,0.000423,0.000185,0.000276,0.000046
MSFT,0.000099,0.000185,0.000287,0.000219,0.000019
NVDA,0.000159,0.000276,0.000219,0.000922,0.000014
WMT,0.000062,0.000046,0.000019,0.000014,0.000218


In [12]:
w_port

array([0.2, 0.2, 0.2, 0.2, 0.2])

**`*`**: Multiplicación escalar

**`@`**: Multiplicación matricial

Varianza (Riesgo)

In [13]:
var_port = w_port.T@cov@w_port 
var_port

np.float64(0.00018300027667894294)

Desviación estándar (Volatilidad)

In [14]:
vol_port = np.sqrt(w_port.T@cov@w_port) 
vol_port

np.float64(0.01352775948481281)

In [15]:
# Volatilidad anualizada
vol_port * np.sqrt(252)

np.float64(0.21474652435625965)

Coeficiente de variación

In [16]:
vol_port / r_port

np.float64(0.041425203275988136)

-----

Usemos la **correlación**, ya que esta sí puede medir la magnitud, su formula es:

$$
\rho_{xy}
=
\frac{\operatorname{Cov}_{xy}}
{\sigma_x \sigma_y}
$$

`.corr()` crea una matriz de correlación es simétrica es cuadrada.

Simétrica: La parte inferior de la diagonal es igual a la parte superior, por lo que una matriz de correlación es igual a su transpuesta.

In [17]:
rets.corr()

Ticker,AAPL,AMZN,MSFT,NVDA,WMT
Ticker,,,,,
AAPL,1.000000,0.361332,0.334405,0.299799,0.241137
AMZN,0.361332,1.000000,0.532321,0.442677,0.150536
MSFT,0.334405,0.532321,1.000000,0.426770,0.076781
NVDA,0.299799,0.442677,0.426770,1.000000,0.032102
WMT,0.241137,0.150536,0.076781,0.032102,1.000000


En estándar de mercado, **una correlación es baja cuando está entre $[-0.25, +0.25]$**